# FP_NB_Build_Semantic_Layer
**Northbrook Lending: semantic model and report as code**

Builds the serving layer with **semantic-link-labs** instead of clicking through the UI, so it's repeatable and reviewable in Git.

1. **`FP_Northbrook_Model`**: a **Direct Lake (on OneLake)** semantic model over the `FP_Gold` Delta tables. No import refresh and no data copies.
2. **Model design**: star-schema relationships, a marked date table, hidden keys, sort orders, descriptions and the DAX measures:
   *Total Lent, Outstanding Balance, Arrears %, Arrears 30+/60+/90+ %, 1–29 / 30–59 / 60–89 / 90+ DPD buckets, Collections Rate, On-time Payment %*, plus DQ and pipeline-operations measures.
3. **`FP_Northbrook_Portfolio`**: a 3-page Power BI report (Portfolio Overview, Arrears & Collections, Data Quality), generated from a report definition.

Balances are **semi-additive**: snapshot measures take the latest month-end inside the current filter, never a sum across months.

In [1]:
# Install semantic-link-labs, the library used to build the semantic model and report in code (-q = quiet output)
%pip install -q semantic-link-labs

StatementMeta(, d789b3fc-78ef-4dd1-89f6-18bcdc9c5165, 7, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: /nfs4/pyenv-5020c4fd-c11f-40f2-aaec-8a695c0145e4/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Settings for this build
model_name = "FP_Northbrook_Model"      # name of the semantic model to create
report_name = "FP_Northbrook_Portfolio"  # name of the Power BI report to create
lakehouse = "FP_Gold"                 # lakehouse whose Delta tables the model reads
overwrite_model = True                 # True = delete and rebuild the model if it already exists

StatementMeta(, d789b3fc-78ef-4dd1-89f6-18bcdc9c5165, 9, Finished, Available, Finished, False)

In [3]:
# ---------- 1. Create the Direct Lake semantic model ----------
# json builds the report definition; uuid makes unique IDs
import json, uuid
# Semantic Link libraries for working with Fabric items and Power BI models
import sempy.fabric as fabric
import sempy_labs as labs
# Direct Lake tools - used to generate the model
from sempy_labs import directlake
# Report tools - used to create the Power BI report
from sempy_labs import report as labs_report
# Tabular Object Model connection - lets the code read and edit the model's tables, columns and measures
from sempy_labs.tom import connect_semantic_model

# Gold tables to include in the model
TABLES = ["dbo.DimDate", "dbo.DimCustomer", "dbo.DimProduct", "dbo.DimBranch",
          "dbo.FactRepayment", "dbo.FactLoanSnapshot", "dbo.AggArrearsCollectionsMonthly",
          "dbo.DQ_Results", "dbo.ETL_RunLog"]

# Create a Direct Lake model over those Gold tables, replacing any existing one if overwrite_model is True
directlake.generate_direct_lake_semantic_model(
    dataset=model_name, tables=TABLES, source=lakehouse, source_type="Lakehouse",
    use_sql_endpoint=False,          # Direct Lake on OneLake
    refresh=False, overwrite=overwrite_model)   # no refresh needed - Direct Lake reads the Delta files directly

# Open the new model read-only and list its tables, as a quick check
with connect_semantic_model(dataset=model_name, readonly=True) as tom:
    print("Tables in model:", [t.Name for t in tom.model.Tables])

StatementMeta(, 2ffb6dfa-ce41-438c-87ef-c02db51fb32a, 10, Finished, Available, Finished, False)

🟢 The 'FP_Northbrook_Model' semantic model was created within the 'FP_Northbrook_Dev' workspace.
Tables in model: ['DimDate', 'DimCustomer', 'DimProduct', 'DimBranch', 'FactRepayment', 'FactLoanSnapshot', 'AggArrearsCollectionsMonthly', 'DQ_Results', 'ETL_RunLog']


In [4]:
# ---------- 2. Relationships, date table, hidden keys, sort orders, descriptions ----------
# Star-schema joins - fact table, fact column, dimension table, dimension column
RELATIONSHIPS = [
    ("FactRepayment", "DueDateKey", "DimDate", "DateKey"),
    ("FactRepayment", "CustomerSK", "DimCustomer", "CustomerSK"),
    ("FactRepayment", "ProductKey", "DimProduct", "ProductKey"),
    ("FactRepayment", "BranchKey", "DimBranch", "BranchKey"),
    ("FactLoanSnapshot", "SnapshotDateKey", "DimDate", "DateKey"),
    ("FactLoanSnapshot", "CustomerSK", "DimCustomer", "CustomerSK"),
    ("FactLoanSnapshot", "ProductKey", "DimProduct", "ProductKey"),
    ("FactLoanSnapshot", "BranchKey", "DimBranch", "BranchKey"),
    ("AggArrearsCollectionsMonthly", "SnapshotDateKey", "DimDate", "DateKey"),
]
# Key and ID columns to hide from report builders - they are only used for joins
HIDE = {
    "FactRepayment": ["DueDateKey", "PaidDateKey", "CustomerSK", "ProductKey", "BranchKey", "RepaymentID"],
    "FactLoanSnapshot": ["SnapshotDateKey", "CustomerSK", "ProductKey", "BranchKey", "LoanID"],
    "AggArrearsCollectionsMonthly": ["SnapshotDateKey"],
    "DimCustomer": ["CustomerSK"], "DimProduct": ["ProductKey"], "DimBranch": ["BranchKey"], "DimDate": ["DateKey"],
}
# Plain-English description for each table, shown to report builders in Power BI
DESCRIPTIONS = {
    "DimDate": "Calendar 2015-2027 with UK financial year (April-March).",
    "DimCustomer": "Customer SCD Type 2 history (one row per version). No direct identifiers.",
    "DimProduct": "Lending products: unsecured, asset finance, secured and mortgage.",
    "DimBranch": "Fictional Northbrook Lending branches.",
    "FactRepayment": "One row per instalment due, with paid/late/missed status. Point-in-time customer key.",
    "FactLoanSnapshot": "Loan x month-end snapshot (last 12 month-ends + as-at date): balance, DPD, arrears bucket.",
    "AggArrearsCollectionsMonthly": "Pre-aggregated arrears and collections summary by month-end and product type.",
    "DQ_Results": "Data quality check results, one row per check per run (FP_NB_DataQuality).",
    "ETL_RunLog": "One row per FP_PL_Daily_Load pipeline run (FP_NB_Pipeline_Logger).",
}

# Open the model for editing
with connect_semantic_model(dataset=model_name, readonly=False) as tom:
    # Add each relationship - many fact rows to one dimension row, filtering from dimension to fact only
    for f_t, f_c, t_t, t_c in RELATIONSHIPS:
        tom.add_relationship(from_table=f_t, from_column=f_c, to_table=t_t, to_column=t_c,
                             from_cardinality="Many", to_cardinality="One",
                             cross_filtering_behavior="OneDirection")
    # Mark DimDate as the date table, so Power BI time-intelligence functions work
    tom.mark_as_date_table(table_name="DimDate", column_name="Date")
    # Hide the key and ID columns listed above
    for table, cols in HIDE.items():
        for c in cols:
            tom.model.Tables[table].Columns[c].IsHidden = True
    # Sort month and day names in calendar order, not alphabetically
    d = tom.model.Tables["DimDate"]
    d.Columns["MonthName"].SortByColumn = d.Columns["MonthNumber"]
    d.Columns["MonthShort"].SortByColumn = d.Columns["MonthNumber"]
    d.Columns["DayName"].SortByColumn = d.Columns["DayOfWeek"]
    # Add the table descriptions
    for table, text in DESCRIPTIONS.items():
        tom.model.Tables[table].Description = text
# Changes are saved to the model when the 'with' block ends
print("Relationships, date table, hidden keys, sort orders and descriptions applied.")

StatementMeta(, 2ffb6dfa-ce41-438c-87ef-c02db51fb32a, 11, Finished, Available, Finished, False)

🟢 The 'DimDate' table has been marked as a date table using the 'Date' column as its primary date key.
Relationships, date table, hidden keys, sort orders and descriptions applied.


In [5]:
# ---------- 3. DAX measures ----------
# Snapshot measures are semi-additive: evaluate at the latest SnapshotDate visible in the current filter context.
# Wrap a DAX expression so it is worked out at the latest snapshot date only
def at_latest_snapshot(expr, extra_filter=""):
    # Add any extra filter, comma-separated, if one was given
    extra = f", {extra_filter}" if extra_filter else ""
    # Find the latest snapshot date in view, then work out the expression for that date only
    return ("VAR _d = MAX ( FactLoanSnapshot[SnapshotDate] )\n"
            f"RETURN CALCULATE ( {expr}, FactLoanSnapshot[SnapshotDate] = _d{extra} )")

# DAX snippet - time of the latest data quality run, ignoring any filters
latest_dq = "VAR _r = CALCULATE ( MAX ( DQ_Results[RunTimestamp] ), ALL ( DQ_Results ) )\n"
# DAX snippet - end time of the latest pipeline run, for the operations measures
latest_run = "VAR _t = CALCULATE ( MAX ( ETL_RunLog[EndTimeUtc] ), ALL ( ETL_RunLog ) )\n"
# Display formats - pounds with no decimals, percentage to 2 places, whole number
GBP, PCT, INT = "£#,0", "0.00%", "#,0"

# Every measure to add to the model
MEASURES = [
    # table, name, DAX, format, folder, description
    # Portfolio measures - lending totals and balances
    ("FactLoanSnapshot", "Total Lent", at_latest_snapshot("SUM ( FactLoanSnapshot[PrincipalAmount] )"), GBP, "Portfolio",
     "Principal advanced on all loans originated up to the latest snapshot in context."),
    ("FactLoanSnapshot", "Outstanding Balance", at_latest_snapshot("SUM ( FactLoanSnapshot[OutstandingBalance] )"), GBP, "Portfolio",
     "Outstanding principal at the latest snapshot in context (semi-additive over time)."),
    ("FactLoanSnapshot", "Open Loans", at_latest_snapshot("SUM ( FactLoanSnapshot[IsOpen] )"), INT, "Portfolio", "Loans not yet fully repaid."),
    ("FactLoanSnapshot", "Average Interest Rate", at_latest_snapshot("AVERAGE ( FactLoanSnapshot[InterestRatePct] )", "FactLoanSnapshot[IsOpen] = 1") , "0.00\"%\"", "Portfolio",
     "Average contractual rate across open loans."),
    # Arrears measures - counts, rates and balances for loans behind on payments
    ("FactLoanSnapshot", "Loans in Arrears", at_latest_snapshot("SUM ( FactLoanSnapshot[IsInArrears] )"), INT, "Arrears", "Open loans 1+ days past due."),
    ("FactLoanSnapshot", "Arrears %", "DIVIDE ( [Loans in Arrears], [Open Loans] )", PCT, "Arrears", "Loans in arrears / open loans."),
    ("FactLoanSnapshot", "Arrears Balance", at_latest_snapshot("SUM ( FactLoanSnapshot[ArrearsAmount] )"), GBP, "Arrears", "Overdue amount owed."),
    ("FactLoanSnapshot", "Arrears Balance %", "DIVIDE ( [Arrears Balance], [Outstanding Balance] )", PCT, "Arrears", "Overdue amount / outstanding balance."),
    ("FactLoanSnapshot", "Arrears 30+ %", "DIVIDE ( " + at_latest_snapshot("SUM ( FactLoanSnapshot[Is30Plus] )") + ", [Open Loans] )", PCT, "Arrears", "Share of open loans 30+ days past due."),
    ("FactLoanSnapshot", "Arrears 60+ %", "DIVIDE ( " + at_latest_snapshot("SUM ( FactLoanSnapshot[Is60Plus] )") + ", [Open Loans] )", PCT, "Arrears", "Share of open loans 60+ days past due."),
    ("FactLoanSnapshot", "Arrears 90+ %", "DIVIDE ( " + at_latest_snapshot("SUM ( FactLoanSnapshot[Is90Plus] )") + ", [Open Loans] )", PCT, "Arrears", "Share of open loans 90+ days past due (default proxy)."),
] + [
    # One 'Loans N DPD' measure per days-past-due bucket, generated in a loop
    ("FactLoanSnapshot", f"Loans {b} DPD", at_latest_snapshot("SUM ( FactLoanSnapshot[IsInArrears] )", f"FactLoanSnapshot[ArrearsBucket] = \"{b}\""), INT,
     "Arrears\\DPD buckets", f"Loans {b} days past due.") for b in ["1-29", "30-59", "60-89", "90+"]
] + [
    # Collections measures - from repayments, so they add up across months normally
    ("FactRepayment", "Amount Due", "SUM ( FactRepayment[DueAmount] )", GBP, "Collections", "Instalments falling due."),
    ("FactRepayment", "Amount Collected", "SUM ( FactRepayment[PaidAmount] )", GBP, "Collections", "Amount received against instalments due."),
    ("FactRepayment", "Collections Rate", "DIVIDE ( [Amount Collected], [Amount Due] )", PCT, "Collections", "Amount collected / amount due."),
    ("FactRepayment", "On-time Payment %", "DIVIDE ( SUM ( FactRepayment[IsOnTime] ), COUNTROWS ( FactRepayment ) )", PCT, "Collections", "Instalments paid on or before the due date."),
    ("FactRepayment", "Missed Payments", "SUM ( FactRepayment[IsMissed] )", INT, "Collections", "Instalments missed or only partly paid."),
    # Data quality measures - check counts, pass rate and failed rows for the latest run only
    ("DQ_Results", "DQ Checks (Latest Run)", latest_dq + "RETURN CALCULATE ( COUNTROWS ( DQ_Results ), DQ_Results[RunTimestamp] = _r )", INT, "Data quality", "Checks executed in the latest DQ run."),
    ("DQ_Results", "DQ Passed (Latest Run)", latest_dq + "RETURN CALCULATE ( COUNTROWS ( DQ_Results ), DQ_Results[RunTimestamp] = _r, DQ_Results[Status] = \"PASS\" )", INT, "Data quality", ""),
    ("DQ_Results", "DQ Failed (Latest Run)", latest_dq + "RETURN CALCULATE ( COUNTROWS ( DQ_Results ), DQ_Results[RunTimestamp] = _r, DQ_Results[Status] = \"FAIL\" ) + 0", INT, "Data quality", ""),
    ("DQ_Results", "DQ Warnings (Latest Run)", latest_dq + "RETURN CALCULATE ( COUNTROWS ( DQ_Results ), DQ_Results[RunTimestamp] = _r, DQ_Results[Status] = \"WARN\" ) + 0", INT, "Data quality", ""),
    ("DQ_Results", "DQ Pass Rate", "DIVIDE ( [DQ Passed (Latest Run)], [DQ Checks (Latest Run)] )", PCT, "Data quality", "Share of checks passing in the latest run."),
    ("DQ_Results", "Failed Rows (Latest Run)", latest_dq + "RETURN IF ( MAX ( DQ_Results[RunTimestamp] ) = _r, SUM ( DQ_Results[FailedRows] ) )", INT, "Data quality",
     "Blank for older runs, so tables show only the latest run."),
    # Operations measures - run count, plus status and duration of the latest pipeline run
    ("ETL_RunLog", "Pipeline Runs", "COUNTROWS ( ETL_RunLog )", INT, "Operations", ""),
    ("ETL_RunLog", "Last Pipeline Status", latest_run + "RETURN CALCULATE ( MAX ( ETL_RunLog[Status] ), ETL_RunLog[EndTimeUtc] = _t )", None, "Operations", ""),
    ("ETL_RunLog", "Last Run Duration (min)", latest_run + "RETURN CALCULATE ( MAX ( ETL_RunLog[DurationSeconds] ), ETL_RunLog[EndTimeUtc] = _t ) / 60", "0.0", "Operations", ""),
]

# Open the model for editing; changes are saved when the 'with' block ends
with connect_semantic_model(dataset=model_name, readonly=False) as tom:
    # Add each measure with its format, display folder and description
    for table, name, expr, fmt, folder, desc in MEASURES:
        tom.add_measure(table_name=table, measure_name=name, expression=expr, format_string=fmt,
                        display_folder=folder, description=desc or None)
print(f"{len(MEASURES)} measures added.")

StatementMeta(, 2ffb6dfa-ce41-438c-87ef-c02db51fb32a, 12, Finished, Available, Finished, False)

29 measures added.


In [6]:
# ---------- 4. Refresh (Direct Lake framing) and validate the measures ----------
# Refresh the model - for Direct Lake this just points it at the latest version of the Delta tables (framing)
labs.refresh_semantic_model(dataset=model_name)
# Run a DAX query that returns the headline measures, to check they calculate without errors
check = fabric.evaluate_dax(model_name, """
EVALUATE ROW (
  "Total Lent", [Total Lent], "Outstanding Balance", [Outstanding Balance], "Open Loans", [Open Loans],
  "Arrears %", [Arrears %], "Arrears 90+ %", [Arrears 90+ %], "Collections Rate", [Collections Rate],
  "DQ Pass Rate", [DQ Pass Rate], "Last Pipeline Status", [Last Pipeline Status] )""")
# Show the results as a one-row table
display(check)

StatementMeta(, 2ffb6dfa-ce41-438c-87ef-c02db51fb32a, 13, Finished, Available, Finished, False)

⌛ Refresh of the 'FP_Northbrook_Model' semantic model within the 'FP_Northbrook_Dev' workspace is in progress...
🟢 Refresh 'full' of the 'FP_Northbrook_Model' semantic model within the 'FP_Northbrook_Dev' workspace is complete.


SynapseWidget(Synapse.DataFrame, 1a5de87b-eb93-4ca3-b539-a3bada3e6fa6)

In [7]:
# ---------- 5. Report definition (PBIR-Legacy report.json) and creation ----------
# json and base64 build and encode the report files; uuid makes visual IDs; time is for waiting between status checks
import json, uuid, base64, time
# Semantic Link - Fabric REST client and item lookups
import sempy.fabric as fabric
# Report tools from semantic-link-labs
from sempy_labs import report as labs_report

# Build one report visual - its type, position (x, y), size (w, h), the fields in each role, and an optional title
def visual(vtype, x, y, w, h, roles, title=None):
    """roles: {role: [(table, field, 'm' | 'c')]} -> one visual container."""
    # Table aliases, query FROM list, query SELECT list, and the fields placed in each role
    aliases, frm, sel, proj = {}, [], [], {}
    # Work through each role and its fields
    for role, fields in roles.items():
        proj[role] = []
        for table, field, kind in fields:
            # Give each table a short alias (t0, t1...) the first time it is used
            if table not in aliases:
                aliases[table] = f"t{len(aliases)}"
                frm.append({"Name": aliases[table], "Entity": table, "Type": 0})
            # Full field name, e.g. FactLoanSnapshot.Total Lent
            ref = f"{table}.{field}"
            key = "Measure" if kind == "m" else "Column"
            # Add the field to the query once, as a measure or a column
            if ref not in [s["Name"] for s in sel]:
                sel.append({key: {"Expression": {"SourceRef": {"Source": aliases[table]}}, "Property": field}, "Name": ref})
            # Put the field into its role, e.g. Category or Y
            proj[role].append({"queryRef": ref})
    # The visual's settings - its type, the fields in each role, and the query that fetches them
    single = {"visualType": vtype, "projections": proj,
              "prototypeQuery": {"Version": 2, "From": frm, "Select": sel}, "drillFilterOtherVisuals": True}
    # Show a title on the visual if one was given
    if title:
        single["vcObjects"] = {"title": [{"properties": {
            "show": {"expr": {"Literal": {"Value": "true"}}},
            "text": {"expr": {"Literal": {"Value": f"'{title}'"}}}}}]}
    # Give the visual a unique 20-character ID and set its position and size
    name = uuid.uuid4().hex[:20]
    cfg = {"name": name, "layouts": [{"id": 0, "position": {"x": x, "y": y, "z": 0, "width": w, "height": h}}],
           "singleVisual": single}
    # Return the visual container - its position and size, with the settings stored as JSON text
    return {"x": x, "y": y, "z": 0, "width": w, "height": h, "config": json.dumps(cfg), "filters": "[]"}

# Shortcuts to tag a field as a measure (M) or a column (C)
M = lambda t, f: (t, f, "m")
C = lambda t, f: (t, f, "c")
# Short names for the tables used most often
FLS, FR, DQ, LOG = "FactLoanSnapshot", "FactRepayment", "DQ_Results", "ETL_RunLog"
# Shortcut for a small KPI card showing one measure, placed along the top row at position x
card = lambda x, m: visual("card", x, 20, 232, 110, {"Values": [m]})

# The three report pages - name, then each visual's type, position (x, y), size (w, h), fields and title
pages = [
    ("Portfolio Overview", [
        # Page 1 - five headline cards across the top
        card(20, M(FLS, "Total Lent")), card(272, M(FLS, "Outstanding Balance")), card(524, M(FLS, "Open Loans")),
        card(776, M(FLS, "Arrears %")), card(1028, M(FLS, "Average Interest Rate")),
        # Charts - balance trend, balance by product, loans by region, product slicer, balance by credit band
        visual("lineChart", 20, 150, 620, 270, {"Category": [C("DimDate", "YearMonth")], "Y": [M(FLS, "Outstanding Balance")]},
               "Outstanding balance by month-end"),
        visual("clusteredBarChart", 660, 150, 600, 270, {"Category": [C("DimProduct", "ProductName")], "Y": [M(FLS, "Outstanding Balance")]},
               "Outstanding balance by product"),
        visual("clusteredColumnChart", 20, 440, 620, 260, {"Category": [C("DimBranch", "Region")], "Y": [M(FLS, "Open Loans")]},
               "Open loans by region"),
        visual("slicer", 660, 440, 260, 260, {"Values": [C("DimProduct", "ProductType")]}, "Product type"),
        visual("donutChart", 940, 440, 320, 260, {"Category": [C("DimCustomer", "CreditBand")], "Y": [M(FLS, "Outstanding Balance")]},
               "Balance by credit band"),
    ]),
    ("Arrears & Collections", [
        # Page 2 - arrears rates, arrears balance and collections rate
        card(20, M(FLS, "Arrears %")), card(272, M(FLS, "Arrears 30+ %")), card(524, M(FLS, "Arrears 90+ %")),
        card(776, M(FLS, "Arrears Balance")), card(1028, M(FR, "Collections Rate")),
        # Charts - loans by DPD bucket, arrears trend, collections trend, arrears by credit band
        visual("clusteredColumnChart", 20, 150, 620, 270, {"Y": [M(FLS, "Loans 1-29 DPD"), M(FLS, "Loans 30-59 DPD"),
                                                                 M(FLS, "Loans 60-89 DPD"), M(FLS, "Loans 90+ DPD")]},
               "Loans in arrears by days-past-due bucket"),
        visual("lineChart", 660, 150, 600, 270, {"Category": [C("DimDate", "YearMonth")],
                                                  "Y": [M(FLS, "Arrears %"), M(FLS, "Arrears 30+ %"), M(FLS, "Arrears 90+ %")]},
               "Arrears trend by month-end"),
        visual("lineChart", 20, 440, 620, 260, {"Category": [C("DimDate", "YearMonth")], "Y": [M(FR, "Collections Rate"), M(FR, "On-time Payment %")]},
               "Collections rate and on-time payments by due month"),
        visual("clusteredBarChart", 660, 440, 600, 260, {"Category": [C("DimCustomer", "CreditBand")], "Y": [M(FLS, "Arrears %")]},
               "Arrears % by credit band"),
    ]),
    ("Data Quality", [
        # Page 3 - DQ pass rate, checks, failures, last pipeline status and duration
        card(20, M(DQ, "DQ Pass Rate")), card(272, M(DQ, "DQ Checks (Latest Run)")), card(524, M(DQ, "DQ Failed (Latest Run)")),
        card(776, M(LOG, "Last Pipeline Status")), card(1028, M(LOG, "Last Run Duration (min)")),
        # Charts and tables - checks by type, latest results, and pipeline run history
        visual("clusteredBarChart", 20, 150, 400, 270, {"Category": [C(DQ, "CheckType")], "Y": [M(DQ, "DQ Checks (Latest Run)")]},
               "Checks by type (latest run)"),
        visual("tableEx", 440, 150, 820, 270, {"Values": [C(DQ, "Layer"), C(DQ, "TableName"), C(DQ, "CheckName"), C(DQ, "Severity"),
                                                           C(DQ, "Status"), M(DQ, "Failed Rows (Latest Run)")]},
               "Latest data quality results"),
        visual("tableEx", 20, 440, 1240, 260, {"Values": [C(LOG, "StartTimeUtc"), C(LOG, "Status"), C(LOG, "DurationSeconds"),
                                                          C(LOG, "SilverRowsLoaded"), C(LOG, "DQChecks"), C(LOG, "DQFail"), C(LOG, "FailedStep")]},
               "Pipeline run history (FP_PL_Daily_Load)"),
    ]),
]

# Assemble report.json - one 1280x720 page (section) per entry in pages, in order
report_json = {
    "config": json.dumps({"version": "5.43", "activeSectionIndex": 0}),
    "layoutOptimization": 0,
    "sections": [{"name": f"ReportSection{i}", "displayName": name, "displayOption": 1, "ordinal": i,
                  "width": 1280, "height": 720, "filters": "[]", "config": "{}", "visualContainers": vcs}
                 for i, (name, vcs) in enumerate(pages)],
}
# A report definition needs a valid theme and report config. Take both from a blank report saved once
# from the UI (FP_Northbrook_Portfolio_Base), add the generated pages, and apply the result via the Fabric REST API.
# REST client signed in as you, and the ID of the current workspace
client = fabric.FabricRestClient()
ws_id = fabric.get_workspace_id()
# Report saved once from the Power BI UI, used only as a source of the theme file
BASE_REPORT = "FP_Northbrook_Portfolio_Base"

# Strip the host from a full Fabric URL, leaving the path the REST client expects
def _path(url):
    return url.split("fabric.microsoft.com", 1)[-1]

def _lro(resp):
    """Wait for a Fabric long-running operation and return its result."""
    # HTTP 202 means accepted but still running in the background
    if resp.status_code == 202:
        # The Location header gives the address to check on the operation's progress
        loc = _path(resp.headers["Location"])
        status = None
        # Check every 2 seconds, for up to 3 minutes, until it has succeeded or failed
        for _ in range(90):
            time.sleep(2)
            status = client.get(loc).json().get("status")
            if status in ("Succeeded", "Failed"):
                break
        # Stop with an error unless it succeeded; otherwise return the result (updateDefinition returns none)
        if status != "Succeeded":
            raise RuntimeError(f"Operation {status}: {client.get(loc).text}")
        return None if "updateDefinition" in resp.url else client.get(loc + "/result")  # update has no result body
    # Any other error response - stop with the API's error message
    if resp.status_code >= 400:
        raise RuntimeError(resp.text)
    return resp

# Look up a report's ID by its display name; None if there is no such report
def _report_id(name):
    df = fabric.list_items(type="Report")
    hit = df[df["Display Name"] == name]
    return hit["Id"].iloc[0] if len(hit) else None

# Download a report's definition and return its files as {path: file}, optionally in a given format
def _definition(report, fmt=None):
    q = f"?format={fmt}" if fmt else ""
    resp = client.post(f"/v1/workspaces/{ws_id}/reports/{_report_id(report)}/getDefinition{q}")
    return {p["path"]: p for p in _lro(resp).json()["definition"]["parts"]}

# Encode bytes, or any object as JSON, to Base64 text - the format the REST API expects for file contents
def _b64(obj):
    data = obj if isinstance(obj, (bytes, bytearray)) else json.dumps(obj).encode("utf-8")
    return base64.b64encode(data).decode()

# 1. The base report (saved from the UI, enhanced PBIR format) supplies the base theme file.
# Download the base report's files and list them
base = _definition(BASE_REPORT)
print("Base definition parts:", list(base))
# Find the theme file in the base report and get its name without the .json ending
theme_path = next(p for p in base if "BaseThemes/" in p)
theme_name = theme_path.rsplit("/", 1)[-1].removesuffix(".json")
# Read the base report's settings and pick out the theme version it was built with
base_rpt = json.loads(base64.b64decode(base["definition/report.json"]["payload"]))
theme_version = base_rpt.get("themeCollection", {}).get("baseTheme", {}).get("reportVersionAtImport")
# Show which theme was found
print("Base theme:", theme_name, theme_version)

# 2. Report.json with the generated pages plus the theme reference a working report needs.
# Report settings pointing at the base theme, including its version if one was found
cfg = {"version": "5.43", "activeSectionIndex": 0,
       "themeCollection": {"baseTheme": {"name": theme_name, "type": 2,
                                         **({"version": theme_version} if theme_version else {})}}}
# Copy the report definition with that config, and register the theme file as a shared resource
rj = dict(report_json, config=json.dumps(cfg),
          resourcePackages=[{"resourcePackage": {"name": "SharedResources", "type": 2, "disabled": False,
                             "items": [{"type": 202, "path": f"BaseThemes/{theme_name}.json", "name": theme_name}]}}])

# Create the report, linked to the model, if it doesn't exist yet
if _report_id(report_name) is None:
    labs_report.create_report_from_reportjson(report=report_name, dataset=model_name, report_json=rj)

# 3. Keep the target report's own dataset binding (definition.pbir); replace report.json; add the theme file.
# Download the target report's current definition in the older PBIR-Legacy format
target = _definition(report_name, "PBIR-Legacy")
print("Target definition parts:", list(target))
new_parts = [
    # The three files - existing model binding, new report.json, and the theme file copied from the base report
    {"path": "definition.pbir", "payload": target["definition.pbir"]["payload"], "payloadType": "InlineBase64"},
    {"path": "report.json", "payload": _b64(rj), "payloadType": "InlineBase64"},
    {"path": f"StaticResources/SharedResources/BaseThemes/{theme_name}.json",
     "payload": base[theme_path]["payload"], "payloadType": "InlineBase64"},
]
# Upload the new definition to the report through the Fabric REST API and wait for it to finish
_lro(client.post(f"/v1/workspaces/{ws_id}/reports/{_report_id(report_name)}/updateDefinition",
                 json={"definition": {"parts": new_parts}}))
# Confirm the report was updated
print("Report definition applied:", report_name)

StatementMeta(, d789b3fc-78ef-4dd1-89f6-18bcdc9c5165, 12, Finished, Available, Finished, False)

/nfs4/pyenv-5020c4fd-c11f-40f2-aaec-8a695c0145e4/lib/python3.13/site-packages/sempy/fabric/_credentials.py:239: FutureWarning: The 'type' parameter is deprecated and will be removed in a future version. Use 'item_type' instead.
  return func(*args, **kwargs)


/nfs4/pyenv-5020c4fd-c11f-40f2-aaec-8a695c0145e4/lib/python3.13/site-packages/sempy/fabric/_credentials.py:239: FutureWarning: The 'type' parameter is deprecated and will be removed in a future version. Use 'item_type' instead.
  return func(*args, **kwargs)
/nfs4/pyenv-5020c4fd-c11f-40f2-aaec-8a695c0145e4/lib/python3.13/site-packages/sempy/fabric/_credentials.py:239: FutureWarning: The 'type' parameter is deprecated and will be removed in a future version. Use 'item_type' instead.
  return func(*args, **kwargs)


/nfs4/pyenv-5020c4fd-c11f-40f2-aaec-8a695c0145e4/lib/python3.13/site-packages/sempy/fabric/_credentials.py:239: FutureWarning: The 'type' parameter is deprecated and will be removed in a future version. Use 'item_type' instead.
  return func(*args, **kwargs)


Report definition applied: FP_Northbrook_Portfolio


In [8]:
# End the notebook and return the model name, report name and number of measures
notebookutils.notebook.exit(json.dumps({"model": model_name, "report": report_name, "measures": len(MEASURES)}))

StatementMeta(, 2ffb6dfa-ce41-438c-87ef-c02db51fb32a, 15, Finished, Available, Finished, False)

ExitValue: {"model": "FP_Northbrook_Model", "report": "FP_Northbrook_Portfolio", "measures": 29}